In [31]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [32]:
BASE_URL = "https://books.toscrape.com/catalogue/page-1.html"

In [33]:
books_data = []

In [34]:
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def extract_books(soup):
    page_books = []
    for book in soup.find_all("article", class_="product_pod"):
        title = book.h3.a["title"]
        price_gbp = float(book.find("p", class_="price_color").text.strip().replace("£", ""))
        price_inr = round(price_gbp * GBP_TO_INR, 2)
        rating_word = book.p["class"][1]
        rating = RATING_MAP.get(rating_word)
        page_books.append({"title": title, "price": f"₹{price_inr:,.2f}", "rating": rating})
    return page_books

In [35]:
def get_next_page_url(soup, current_url):
    next_li = soup.find("li", class_="next")
    if next_li is None:
        return None
    return urljoin(current_url, next_li.a["href"])

In [36]:
url = BASE_URL
GBP_TO_INR = 129.7  # update to the current rate when you run this
while url:
    response = requests.get(url)
    response.encoding = "utf-8"          # fixes the £ → Â£ mojibake
    soup = BeautifulSoup(response.text, "html.parser")
    books_data.extend(extract_books(soup))
    url = get_next_page_url(soup, url)

print(f"Scraped {len(books_data)} books")

Scraped 1000 books


In [37]:
import pandas as pd
df = pd.DataFrame(books_data)
df.head()

,title,price,rating
0,A Light in the Attic,"₹6,714.57",3
1,Tipping the Velvet,"₹6,970.08",1
2,Soumission,"₹6,497.97",1
3,Sharp Objects,"₹6,202.25",4
4,Sapiens: A Brief History of Humankind,"₹7,033.63",5


In [43]:
df.to_csv("books.csv", index=False)

In [41]:
import os
print(os.getcwd())

C:\Users\Chirantana D Naik\Videos\Data Science\Lab-1
